In [ ]:
# Cell 1: Install system dependencies
!apt-get install -y bedtools samtools

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libhts3 libhtscodecs2
Suggested packages:
  cwltool
The following NEW packages will be installed:
  bedtools libhts3 libhtscodecs2 samtools
0 upgraded, 4 newly installed, 0 to remove and 2 not upgraded.
Need to get 1,525 kB of archives.
After this operation, 3,818 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 bedtools amd64 2.30.0+dfsg-2ubuntu0.1 [563 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhtscodecs2 amd64 1.1.1-3 [53.2 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhts3 amd64 1.13+ds-2build1 [390 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 samtools amd64 1.13-4 [520 kB]
Fetched 1,525 kB in 2s (638 kB/s)
Selecting previously unselected package bedtools.
(Reading database ... 118194 files and directories cur

In [ ]:
# Cell 2: Mount Google Drive (stores reference genome permanently)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cell 3: Download hg38 reference genome (one‑time operation)
# Will skip if file already exists.
%%bash
mkdir -p /content/drive/MyDrive/ML_Project/data
cd /content/drive/MyDrive/ML_Project/data

if [ ! -f hg38.fa ]; then
    echo "Downloading hg38.fa.gz (~1 GB) ..."
    wget -q http://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
    echo "Decompressing ..."
    gunzip -k hg38.fa.gz   # -k keeps the .gz if you want to save space remove it
fi

# Create FASTA index (required by bedtools)
if [ ! -f hg38.fa.fai ]; then
    echo "Indexing hg38.fa ..."
    samtools faidx hg38.fa
fi

In [ ]:
!ls -lh /content/drive/MyDrive/ML_Project/data/

total 4.0G
-rw------- 1 root root  12K Apr 12 09:15 hg38.chrom.sizes
-rw------- 1 root root 3.1G Jan 16  2014 hg38.fa
-rw------- 1 root root  19K Apr 12 09:07 hg38.fa.fai
-rw------- 1 root root 939M Jan 16  2014 hg38.fa.gz
-rw------- 1 root root 572K Apr 12 09:14 sp1_centered_101bp.bed
-rw------- 1 root root 2.0M Apr 12 09:44 sp1_dataset_ready.npz
-rw------- 1 root root 582K Apr 12 09:17 sp1_negative_101bp.bed
-rw------- 1 root root 2.9M Apr 12 09:33 sp1_negative_101bp_clean.fasta
-rw------- 1 root root 3.0M Apr 12 09:17 sp1_negative_101bp.fasta
-rw------- 1 root root 3.0M Apr 12 09:33 sp1_positive_101bp_clean.fasta
-rw------- 1 root root 3.0M Apr 12 09:17 sp1_positive_101bp.fasta
-rw------- 1 root root 1.5M Apr 12 09:11 sp1_raw_data.narrowPeak.bed


In [ ]:
# Cell 4: Parse narrowPeak file, center 101 bp on summit, filter invalid coords
import pandas as pd

# Path to ENCODE narrowPeak file on Drive
raw_file_path = '/content/drive/MyDrive/ML_Project/data/sp1_raw_data.narrowPeak.bed'

# ENCODE narrowPeak columns (BED6+4 format)
columns = ['chrom', 'start', 'end', 'name', 'score', 'strand',
           'signal', 'pval', 'qval', 'peak']

df = pd.read_csv(raw_file_path, sep='\t', header=None, names=columns)
df[['start', 'end', 'peak']] = df[['start', 'end', 'peak']].astype(int)

# Center 101 bp window on summit
df['summit_abs'] = df['start'] + df['peak']
df['new_start'] = df['summit_abs'] - 50
df['new_end']   = df['summit_abs'] + 51  # half-open interval [start, end)

# Filter: remove peaks too close to chromosome boundaries
initial_count = len(df)
df = df[df['new_start'] >= 0]
filtered_count = len(df)
print(f"Removed {initial_count - filtered_count} peaks near chromosome boundaries.")

# Save as BED3 format
bed_out = '/content/drive/MyDrive/ML_Project/data/sp1_centered_101bp.bed'
df[['chrom', 'new_start', 'new_end']].to_csv(bed_out, sep='\t', header=False, index=False)

print(f"Processed: {len(df)} intervals")
print(f"Saved to: {bed_out}")
print("\nFirst 5 rows:")
print(df[['chrom', 'new_start', 'new_end']].head())

In [ ]:
# Cell 5: Extract chromosome sizes from FASTA index
%%bash
cd /content/drive/MyDrive/ML_Project/data

# Extract columns 1 (chrom) and 2 (length) from .fai index
cut -f 1,2 hg38.fa.fai > hg38.chrom.sizes

echo "Generated hg38.chrom.sizes. Sample:"
head -n 3 hg38.chrom.sizes

In [ ]:
# Cell 6: Generate positive FASTA and create matched negative set
%%bash
cd /content/drive/MyDrive/ML_Project/data

echo "=== Step 1/3: Extract positive sequences (Label 1) ==="
bedtools getfasta -fi hg38.fa -bed sp1_centered_101bp.bed -fo sp1_positive_101bp.fasta

echo "=== Step 2/3: Generate negative coordinates (Label 0) ==="
# Shuffle peaks to random genomic locations
# -excl: exclude positive regions
# -noOverlapping: prevent negative regions from overlapping each other
bedtools shuffle \
    -i sp1_centered_101bp.bed \
    -g hg38.chrom.sizes \
    -excl sp1_centered_101bp.bed \
    -noOverlapping \
    -seed 42 \
    > sp1_negative_101bp.bed

echo "=== Step 3/3: Extract negative sequences (Label 0) ==="
bedtools getfasta -fi hg38.fa -bed sp1_negative_101bp.bed -fo sp1_negative_101bp.fasta

echo ""
echo "=== Pipeline Complete ==="
pos_count=$(grep -c "^>" sp1_positive_101bp.fasta)
neg_count=$(grep -c "^>" sp1_negative_101bp.fasta)
echo "Positive sequences: $pos_count"
echo "Negative sequences: $neg_count"
echo "Class balance: 1:1 ✓"

In [ ]:
# Cell 7: Filter sequences containing 'N', standardize to uppercase
import os

def clean_fasta(input_path, output_path):
    """
    Read FASTA, remove sequences containing 'N', uppercase valid ones.
    Handles multi-line sequences correctly.

    Args:
        input_path: Raw FASTA file
        output_path: Cleaned FASTA file
    """
    total = 0
    valid = 0
    discarded = 0

    with open(input_path, 'r') as fin, open(output_path, 'w') as fout:
        header = None
        seq_parts = []

        for line in fin:
            line = line.strip()
            if not line:
                continue

            if line.startswith('>'):
                # Process previous sequence before starting new one
                if header is not None:
                    total += 1
                    full_seq = ''.join(seq_parts).upper()
                    if 'N' not in full_seq:
                        fout.write(f"{header}\n{full_seq}\n")
                        valid += 1
                    else:
                        discarded += 1

                # Start new sequence
                header = line
                seq_parts = []
            else:
                seq_parts.append(line)

        # Don't forget the last sequence
        if header is not None:
            total += 1
            full_seq = ''.join(seq_parts).upper()
            if 'N' not in full_seq:
                fout.write(f"{header}\n{full_seq}\n")
                valid += 1
            else:
                discarded += 1

    print(f"File: {os.path.basename(input_path)}")
    print(f"  Total sequences : {total}")
    print(f"  Discarded (has N): {discarded}")
    print(f"  Retained (clean) : {valid}")
    print(f"  Saved to: {output_path}\n")

    return total, valid, discarded

# --- Execute ---
base_dir = "/content/drive/MyDrive/ML_Project/data"
pos_raw = f"{base_dir}/sp1_positive_101bp.fasta"
neg_raw = f"{base_dir}/sp1_negative_101bp.fasta"
pos_clean = f"{base_dir}/sp1_positive_101bp_clean.fasta"
neg_clean = f"{base_dir}/sp1_negative_101bp_clean.fasta"

print("=== CLEANING PIPELINE ===\n")
t1, v1, d1 = clean_fasta(pos_raw, pos_clean)
t2, v2, d2 = clean_fasta(neg_raw, neg_clean)

print("=== SUMMARY ===")
print(f"Positive retained: {v1}/{t1} ({v1/t1*100:.1f}%)")
print(f"Negative retained: {v2}/{t2} ({v2/t2*100:.1f}%)")

# Check class balance after cleaning
if v1 != v2:
    print(f"\n⚠️  Warning: Class imbalance ({v1} vs {v2})")
    print("   Consider downsampling majority class for balanced training.")
else:
    print(f"\n✅ Classes balanced: {v1} each")